# 📉 Cross-Entropy Loss & Perplexity: Complete Hands-On Lab

Welcome! In this notebook, we master <b>Cross-Entropy Loss</b> and <b>Perplexity (PPL)</b>—the core metrics that guide foundation model pretraining.

---

## 1. The Core Metaphor: The 100-Sided Die

Imagine an untrained model trying to predict the next word in a code file.<br>
Because its weights are randomly initialized, it knows nothing! It is like rolling a fair <b>V-sided die</b> (where V is the vocabulary size).

<pre style='font-size: 14px; background: #0d1117; color: #c9d1d9; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
Cross-Entropy Loss asks: "How surprised was the model by the real word?"

Loss = -ln( Probability of Correct Word )

1. Perfect Prediction: Model gives 100% probability (P = 1.0) &rarr; Loss = -ln(1.0) = 0.0
2. Random Guessing:   Model gives 1/V probability           &rarr; Loss = -ln(1/V) = ln(V)
</pre>

---
## 2. Visual Pipeline: Logits &rarr; Softmax &rarr; Cross-Entropy

<pre style='font-size: 13px; background: #0d1117; color: #58a6ff; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
1. Raw Model Output (Logits):      [  2.1,   -0.5,   1.2,   -1.0 ]
                                               │
                                               ▼
2. Softmax (Turn into 100% Probs): [ 0.65,   0.05,  0.27,   0.03 ]
                                      def   class  import  return
                                               │
                                               ▼
3. Ground Truth Target:            "def" (Target Index = 0)
                                               │
                                               ▼
4. Negative Log Likelihood:        Loss = -ln(0.65) = 0.4308
</pre>

---
## 3. What is Perplexity (PPL)?

In research papers, researchers report <b>Perplexity</b>:
<pre style='font-size: 15px; background: #161b22; color: #7ee787; padding: 12px; border-radius: 8px;'>
Perplexity = e^(Loss)
</pre>

<b>In plain English:</b> Perplexity is the <i>effective number of words the model is hesitating between</i>.
<ul>
  <li>If Perplexity = 100,000: Model is guessing blindly across 100,000 words.</li>
  <li>If Perplexity = 10: Model has narrowed it down to 10 likely candidates.</li>
  <li>If Perplexity = 1.0: Model is 100% certain of the correct word.</li>
</ul>

---
## 4. Exercise 1: Paper & Pencil Calculation 📝

Suppose our vocabulary has <b>V = 4 words</b>: <code>['def', 'class', 'import', 'return']</code>.

### Case A: Untrained Model (Random Guessing)
- Probability for all 4 words = 1 / 4 = <b>0.25</b>
- <b>Loss = -ln(0.25) ≈ 1.3863</b> &nbsp; (Notice: ln(4) = 1.3863)
- <b>Perplexity = e^(1.3863) = 4.0</b> &nbsp; (Hesitating between all 4 words)

### Case B: Trained Model (Confident Guess)
- Probability for the correct word = <b>0.80</b>
- <b>Loss = -ln(0.80) ≈ 0.2231</b>
- <b>Perplexity = e^(0.2231) ≈ 1.25</b> &nbsp; (Almost completely certain!)

In [1]:
import math
import torch
import torch.nn.functional as F

# Case A: Equal probabilities across 4 classes
logits_equal = torch.tensor([[1.0, 1.0, 1.0, 1.0]])
target = torch.tensor([0])

loss_equal = F.cross_entropy(logits_equal, target)
ppl_equal = torch.exp(loss_equal)

print(
    f"Case A (Untrained) Loss:       {loss_equal.item():.4f} (Matches ln(4) = {math.log(4):.4f})"
)
print(
    f"Case A (Untrained) Perplexity: {ppl_equal.item():.2f} (Hesitating between 4 words)"
)

# Case B: Confident model
# High logit on index 0
logits_confident = torch.tensor([[4.0, 0.5, 0.2, -1.0]])
loss_confident = F.cross_entropy(logits_confident, target)
ppl_confident = torch.exp(loss_confident)

print(f"\nCase B (Trained) Loss:         {loss_confident.item():.4f}")
print(
    f"Case B (Trained) Perplexity:   {ppl_confident.item():.2f} (Near 1.0 certainty!)"
)

Case A (Untrained) Loss:       1.3863 (Matches ln(4) = 1.3863)
Case A (Untrained) Perplexity: 4.00 (Hesitating between 4 words)

Case B (Trained) Loss:         0.0576
Case B (Trained) Perplexity:   1.06 (Near 1.0 certainty!)


---
## 5. The Foundation Model Step-0 Sanity Check 🔍

Before launching any large training run on your DGX GPU, we always check Step 0 loss.<br>
For our model's vocabulary of <b>100,262 tokens</b>, the initial loss must equal:

<pre style='font-size: 15px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
Expected Step 0 Loss = ln(100,262) ≈ 11.5155
</pre>

Let's verify this on a simulated batch of uninitialized logits:

In [2]:
vocab_size = 100262
expected_initial_loss = math.log(vocab_size)

# Random uninitialized logits from a standard normal distribution
torch.manual_seed(42)
dummy_logits = torch.randn(2, vocab_size) * 0.02  # standard 0.02 init
dummy_targets = torch.randint(0, vocab_size, (2,))

actual_initial_loss = F.cross_entropy(dummy_logits, dummy_targets).item()

print(f"Expected Theoretical Loss: {expected_initial_loss:.4f}")
print(f"Actual Initialized Loss:   {actual_initial_loss:.4f}")
print(
    f"Difference:                {abs(actual_initial_loss - expected_initial_loss):.4f}"
)

assert abs(actual_initial_loss - expected_initial_loss) < 0.05
print("\nSUCCESS: Step 0 initialization sanity check verified!")

Expected Theoretical Loss: 11.5155
Actual Initialized Loss:   11.5202
Difference:                0.0047

SUCCESS: Step 0 initialization sanity check verified!
